# Employee Turnover Analytics
## Portobello Tech — HR Department ML Pipeline

This notebook implements the complete 7-step machine learning pipeline to predict
employee turnover and recommend targeted retention strategies.

### Pipeline Steps
1. **Data Quality Checks** — missing values, types, duplicates, value ranges
2. **Exploratory Data Analysis** — correlation heatmap, distributions, bar charts
3. **Clustering** — K-Means (k=3) on employees who left
4. **Preprocessing & Class Imbalance** — encoding, stratified split, SMOTE
5. **Model Training** — Logistic Regression, Random Forest, Gradient Boosting (5-fold CV)
6. **Model Evaluation** — ROC/AUC, confusion matrices, best model selection
7. **Retention Strategies** — risk zones and targeted recommendations

### Cell 1 — Environment Setup & Imports

**Purpose:**
Prepares the Python environment for the entire analysis. Every library,
path, and the logger must be initialised once here before any other cell
can run.

**What this cell does — step by step:**
1. Adds the **project root** folder to Python's module search path so
   `import src.*` works correctly from inside the `notebooks/` subfolder.
2. Suppresses noisy library warnings to keep notebook output readable.
3. Sets matplotlib to **headless (Agg) mode** — figures are saved to disk
   instead of being rendered interactively, which avoids display errors on
   Windows.
4. Imports all third-party libraries (`pandas`, `numpy`, `matplotlib`,
   `seaborn`) and all seven project modules from `src/`.
5. Defines four key **paths** used throughout the notebook:
   - `DATA_PATH` — the raw CSV dataset
   - `OUTPUT_DIR` — root for all generated files
   - `LOG_DIR` — where rotating log files are written
6. Calls `setup_logging()` to start writing a dated log file
   (`outputs/logs/employee_turnover_YYYYMMDD.log`).

**Inputs:** None — this is the bootstrapping cell.

**Outputs / Variables created:**

| Variable | Type | Description |
|----------|------|-------------|
| `PROJECT_ROOT` | `Path` | Absolute path to the repository root |
| `DATA_PATH` | `Path` | Path to `Dataset/HR_comma_sep.csv` |
| `OUTPUT_DIR` | `Path` | Path to `outputs/` |
| `LOG_DIR` | `Path` | Path to `outputs/logs/` |
| `logger` | `Logger` | Root logger — writes to file + console |

> **Run this cell first.** All subsequent cells depend on the imports and
> paths defined here.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 1 — Environment Setup & Imports
# ─────────────────────────────────────────────────────────────────────────────
import sys
import warnings
from pathlib import Path

# Add project root to path so src/ is importable
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')   # Headless backend — saves figures to disk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Project modules
from src.utils.logger_config import setup_logging
from src.utils.exceptions import EmployeeTurnoverError
from src.data_quality.data_quality_checker import DataQualityChecker
from src.eda.exploratory_analyzer import ExploratoryAnalyzer
from src.clustering.employee_clusterer import EmployeeClusterer
from src.preprocessing.data_preprocessor import DataPreprocessor
from src.modeling.model_trainer import ModelTrainer
from src.modeling.model_evaluator import ModelEvaluator
from src.retention.retention_advisor import RetentionAdvisor, ZONE_ORDER

# ─── Paths ───────────────────────────────────────────────────────────────────
DATA_PATH   = PROJECT_ROOT / 'Dataset' / 'HR_comma_sep.csv'
OUTPUT_DIR  = PROJECT_ROOT / 'outputs'
LOG_DIR     = OUTPUT_DIR / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

# ─── Logging ─────────────────────────────────────────────────────────────────
logger = setup_logging(log_dir=LOG_DIR)

print(f'Project root : {PROJECT_ROOT}')
print(f'Data path    : {DATA_PATH}')
print(f'Output dir   : {OUTPUT_DIR}')
print(f'Log dir      : {LOG_DIR}')

---
## Step 1: Data Quality Checks

Before any analysis, we validate the raw dataset:
- **Missing values** — null counts and percentages per column
- **Data types** — verify numeric vs categorical columns
- **Duplicates** — identify exact duplicate rows
- **Value ranges** — satisfaction/evaluation must be [0,1]; left/accident must be {0,1}

### Cell 3 — Load Raw Dataset

**Purpose:**
Reads the HR CSV file from disk into a pandas DataFrame and prints a
quick sanity-check (shape and first 5 rows) so you can confirm the data
loaded correctly before any analysis begins.

**What this cell does — step by step:**
1. Creates a `DataQualityChecker` object pointing at `HR_comma_sep.csv`.
2. Calls `checker.load_data()` which reads the CSV with UTF-8 encoding and
   validates that the file exists and is non-empty.
3. Prints the **shape** (rows × columns) and the **column names** of the
   loaded DataFrame.
4. Displays the **first 5 rows** as a formatted table.

**Inputs:**

| Variable | Type | Description |
| ---------- | ---------- | ------------- |
| `DATA_PATH` | `Path` | Path to `HR_comma_sep.csv` (set in Cell 1) |

**Outputs / Variables created:**

| Variable | Type | Description |
|----------|------|-------------|
| `checker` | `DataQualityChecker` | Holds the loaded DataFrame; reused in Cells 4–7 |
| `df` | `pd.DataFrame` | 14,999 rows × 10 columns of raw HR data |

**Expected output:** Shape `(14999, 10)` with columns: `satisfaction_level`,
`last_evaluation`, `number_project`, `average_montly_hours`,
`time_spend_company`, `Work_accident`, `left`, `promotion_last_5years`,
`sales`, `salary`.

In [ ]:
# ─── Load Data ────────────────────────────────────────────────────────────────
checker = DataQualityChecker(str(DATA_PATH))
df = checker.load_data()

print(f'Dataset shape: {df.shape}')
print(f'\nColumn names: {list(df.columns)}')
print(f'\nFirst 5 rows:')
df.head()

### Cell 4 — Missing Values Analysis

**Purpose:**
Checks every column for null / NaN values. A dataset with missing values
can silently corrupt model training, so this check must pass before
proceeding.

**What this cell does — step by step:**
1. Calls `checker.check_missing_values()` which counts `NaN` entries
   per column using `df.isnull().sum()`.
2. Computes the **percentage** of missing values relative to total rows.
3. Prints a table with `missing_count` and `missing_percentage` per column.
4. Prints the total number of missing values across the entire dataset.

**Inputs:**

| Variable | Type | Description |
| ---------- | ---------- | ------------- |
| `checker` | `DataQualityChecker` | `DataQualityChecker` with loaded DataFrame (from Cell 3) |

**Outputs / Variables created:**

| Variable | Type | Description |
|----------|------|-------------|
| `missing` | `pd.DataFrame` | Columns: `missing_count`, `missing_percentage`; index = column names |

**Expected outcome:** All `missing_count` values should be **0** —
the HR dataset is clean with no missing values.

In [ ]:
# ─── Missing Values ───────────────────────────────────────────────────────────
missing = checker.check_missing_values()
print('Missing Values Analysis:')
print(missing)
print(f'\nTotal missing values: {missing["missing_count"].sum()}')

### Cell 5 — Column Data Types Inspection

**Purpose:**
Verifies that each column holds the correct data type (numeric vs. string).
Wrong types (e.g., `satisfaction_level` stored as `object`) would cause
downstream encoding and model training to fail.

**What this cell does — step by step:**
1. Calls `checker.check_data_types()` which reads `df.dtypes`.
2. Returns and prints a one-column DataFrame mapping column name → dtype.

**Inputs:**

| Variable | Type | Description |
| ---------- | ---------- | ------------- |
| `checker` | `DataQualityChecker` | `DataQualityChecker` with loaded DataFrame |

**Outputs / Variables created:**

| Variable | Type | Description |
|----------|------|-------------|
| `dtypes` | `pd.DataFrame` | Single column `dtype`; index = column names |

**Expected types:**
- `float64` — `satisfaction_level`, `last_evaluation`
- `int64` — `number_project`, `average_montly_hours`, `time_spend_company`,
  `Work_accident`, `left`, `promotion_last_5years`
- `object` (string) — `sales`, `salary`

In [ ]:
# ─── Data Types ──────────────────────────────────────────────────────────────
dtypes = checker.check_data_types()
print('Column Data Types:')
print(dtypes)

### Cell 6 — Duplicate Rows & Value Range Validation

**Purpose:**
Detects exact duplicate rows (copy-paste data errors) and validates that
every column's values fall within their expected domain. For example,
`satisfaction_level` must be between 0 and 1 — a value of 1.5 would be a
data error.

**What this cell does — step by step:**
1. Calls `checker.check_duplicates()` → counts rows where every field is
   identical to another row using `df.duplicated().sum()`.
2. Calls `checker.check_value_ranges()` which applies these rules:
   - `satisfaction_level`, `last_evaluation` → must be in **[0, 1]**
   - `Work_accident`, `left`, `promotion_last_5years` → must be **{0, 1}**
   - `salary` → must be in **{low, medium, high}**
3. Prints a status line (`✓ VALID` / `✗ INVALID`) and invalid count for
   each column.

**Inputs:**

| Variable | Type | Description |
| ---------- | ---------- | ------------- |
| `checker` | `DataQualityChecker` | `DataQualityChecker` with loaded DataFrame |

**Outputs / Variables created:**

| Variable | Type | Description |
|----------|------|-------------|
| `dup_count` | `int` | Number of exact duplicate rows found |
| `ranges` | `dict` | Keys = column names; values = `{valid, min, max, invalid_count}` |

**Expected outcome:** All ranges `✓ VALID`; `dup_count` may be > 0 but
the pipeline handles this gracefully.

In [ ]:
# ─── Duplicates & Value Ranges ───────────────────────────────────────────────
dup_count = checker.check_duplicates()
print(f'Duplicate rows: {dup_count}')

ranges = checker.check_value_ranges()
print('\nValue Range Validation:')
for col, result in ranges.items():
    status = '✓ VALID' if result['valid'] else '✗ INVALID'
    print(f'  {col:30s}: {status}  (invalid_count={result["invalid_count"]})')

### Cell 7 — Consolidated Data Quality Report

**Purpose:**
Aggregates all four quality checks (missing values, data types, duplicates,
value ranges) into a single summary dictionary and prints a human-readable
dashboard. This is the **go / no-go gate** before any analysis begins.

**What this cell does — step by step:**
1. Calls `checker.generate_quality_report()` which internally runs all
   four checks and collects their results.
2. Extracts and prints five key summary metrics.

**Inputs:**

| Variable | Type | Description |
| ---------- | ---------- | ------------- |
| `checker` | `DataQualityChecker` | `DataQualityChecker` with loaded DataFrame |

**Outputs / Variables created:**

| Variable | Type | Description |
|----------|------|-------------|
| `report` | `dict` | Keys: `total_rows`, `total_columns`, `duplicate_count`, `missing_values`, `value_ranges`, `column_names` |

**Interpreting the output:**

| Metric | Good sign |
|--------|-----------|
| Total rows | 14,999 |
| Total columns | 10 |
| Duplicate rows | Any number (noted, not removed) |
| Cols with missing | 0 |
| All ranges valid | True |

In [ ]:
# ─── Full Quality Report ─────────────────────────────────────────────────────
report = checker.generate_quality_report()
print('Quality Report Summary')
print('=' * 40)
print(f'  Total rows         : {report["total_rows"]}')
print(f'  Total columns      : {report["total_columns"]}')
print(f'  Duplicate rows     : {report["duplicate_count"]}')
print(f'  Cols with missing  : {(report["missing_values"]["missing_count"] > 0).sum()}')
print(f'  All ranges valid   : {all(v["valid"] for v in report["value_ranges"].values())}')

---
## Step 2: Exploratory Data Analysis (EDA)

We generate four required visualisations:
- **2.1** Correlation heatmap of all numeric features
- **2.2** Distribution plots for satisfaction_level, last_evaluation, average_montly_hours
- **2.3** Bar chart: project count segmented by left/stayed

All plots are saved to `outputs/plots/eda/`.

### Cell 9 — EDA Initialisation

**Purpose:**
Creates the `ExploratoryAnalyzer` object which will generate all EDA
visualisations, and confirms the output directory where plots will be saved.

**What this cell does — step by step:**
1. Defines `EDA_OUTPUT` as `outputs/plots/eda/` — all EDA PNGs go here.
2. Instantiates `ExploratoryAnalyzer(df, output_dir=EDA_OUTPUT)` which
   stores an immutable copy of `df` and creates the output directory.
3. Prints the resolved output path for confirmation.

**Inputs:**

| Variable | Type | Description |
| ---------- | ---------- | ------------- |
| `df` | `pd.DataFrame` | Validated HR DataFrame (from Cell 3) |
| `OUTPUT_DIR` | `Path` | Root output path (from Cell 1) |

**Outputs / Variables created:**

| Variable | Type | Description |
|----------|------|-------------|
| `EDA_OUTPUT` | `Path` | Directory where all EDA plots are saved |
| `analyzer` | `ExploratoryAnalyzer` | Reused across Cells 10–13 |

In [ ]:
# ─── EDA Initialisation ───────────────────────────────────────────────────────
EDA_OUTPUT = OUTPUT_DIR / 'plots' / 'eda'
analyzer = ExploratoryAnalyzer(df, output_dir=EDA_OUTPUT)

print(f'EDA outputs will be saved to: {EDA_OUTPUT}')

### Cell 10 — Correlation Heatmap (EDA Step 2.1)

**Purpose:**
Visualises the **Pearson correlation** between every pair of numeric
features. This reveals which features move together — high positive
correlation means two features carry similar information, while a strong
correlation with the target column `left` highlights key predictors.

**What this cell does — step by step:**
1. Calls `analyzer.plot_correlation_heatmap()` which:
   a. Selects only numeric columns (`select_dtypes(include='number')`).
   b. Computes the Pearson correlation matrix (`df.corr()`).
   c. Renders an annotated seaborn heatmap with a `coolwarm` colour scale.
   d. Saves the figure to `outputs/plots/eda/correlation_heatmap.png`.
2. Displays the saved PNG inline using `IPython.display.Image`.

**Inputs:**

| Variable | Type | Description |
| ---------- | ---------- | ------------- |
| `analyzer` | `ExploratoryAnalyzer` | `ExploratoryAnalyzer` instance (from Cell 9) |

**Outputs:**

| Output | Location |
|--------|----------|
| `correlation_heatmap.png` | `outputs/plots/eda/` |
| Inline display | Notebook cell output |

**Key insight to look for:** `satisfaction_level` has a strong **negative**
correlation with `left` — employees with low satisfaction are far more
likely to leave.

In [ ]:
# ─── 2.1 Correlation Heatmap ─────────────────────────────────────────────────
analyzer.plot_correlation_heatmap()
print('Correlation heatmap saved.')

# Display inline
from IPython.display import Image, display
display(Image(filename=str(EDA_OUTPUT / 'correlation_heatmap.png')))

### Cell 11 — Distribution Plots (EDA Step 2.2)

**Purpose:**
Shows the **statistical distribution** of three continuous features:
`satisfaction_level`, `last_evaluation`, and `average_montly_hours`.
Histogram + KDE (Kernel Density Estimate) plots reveal whether distributions
are normal, skewed, or bimodal — all of which affect model interpretation.

**What this cell does — step by step:**
1. Calls `analyzer.plot_all_distributions()` which creates a 1×3 subplot
   figure, plots a histogram with an overlaid KDE curve for each of the
   three columns, and saves the combined figure.

**Inputs:**

| Variable | Type | Description |
| ---------- | ---------- | ------------- |
| `analyzer` | `ExploratoryAnalyzer` | `ExploratoryAnalyzer` instance |

**Outputs:**

| Output | Location |
|--------|----------|
| `all_distributions.png` | `outputs/plots/eda/` |
| Inline display | Notebook cell output |

**Key insights to look for:**
- `satisfaction_level` — bimodal peak: many employees are either very
  satisfied or very dissatisfied.
- `average_montly_hours` — another bimodal shape: a cluster around 150 hrs
  (normal workload) and another near 260 hrs (overworked).

In [ ]:
# ─── 2.2 Distribution Plots ───────────────────────────────────────────────────
analyzer.plot_all_distributions()
print('Distribution plots saved.')

display(Image(filename=str(EDA_OUTPUT / 'all_distributions.png')))

### Cell 12 — Project Count Bar Chart (EDA Step 2.3)

**Purpose:**
Compares how many projects employees handle, split by whether they stayed
or left. This reveals whether **under-utilisation** (too few projects) or
**overload** (too many projects) is associated with higher turnover.

**What this cell does — step by step:**
1. Calls `analyzer.plot_project_count_bar()` which uses `sns.countplot`
   with `x='number_project'` and `hue='left'` to draw side-by-side bars.
2. Saves the figure to disk.
3. Displays it inline.
4. Prints the key inference from the chart.

**Inputs:**

| Variable | Type | Description |
| ---------- | ---------- | ------------- |
| `analyzer` | `ExploratoryAnalyzer` | `ExploratoryAnalyzer` instance |

**Outputs:**

| Output | Location |
|--------|----------|
| `project_count_bar.png` | `outputs/plots/eda/` |
| Inline display | Notebook cell output |

**Key insight:** Employees with **2 projects** (under-utilised) and those
with **6–7 projects** (burned out) show disproportionately high turnover.

In [ ]:
# ─── 2.3 Project Count Bar Chart ─────────────────────────────────────────────
analyzer.plot_project_count_bar()
print('Project count bar chart saved.')

display(Image(filename=str(EDA_OUTPUT / 'project_count_bar.png')))

print('\nInference: Employees with only 2 projects or 6-7 projects show the')
print('highest turnover rates, indicating under-utilisation or burn-out.')

### Cell 13 — Turnover Rate by Salary & Department

**Purpose:**
Computes the actual **percentage of employees who left**, grouped by salary
band and by department. This contextualises the model by showing which
HR segments have the highest raw attrition rates.

**What this cell does — step by step:**
1. Calls `analyzer.compute_turnover_rate_by_feature('salary')` which
   groups `df` by salary level and computes:
   `turnover_rate = left_count / total_count` per group.
2. Repeats for `'sales'` (the department column).
3. Prints both result DataFrames.

**Inputs:**

| Variable | Type | Description |
| ---------- | ---------- | ------------- |
| `analyzer` | `ExploratoryAnalyzer` | `ExploratoryAnalyzer` instance |

**Outputs:** Two DataFrames printed to output (not stored in variables).

| Column | Meaning |
|--------|---------|
| `count` | Total employees in this group |
| `left_count` | Number who left |
| `turnover_rate` | Fraction who left (0.0 – 1.0) |

**Key insight:** `low` salary employees consistently show higher turnover
rates than `medium` or `high` salary employees.

In [ ]:
# ─── Turnover Rate by Feature ─────────────────────────────────────────────────
print('Turnover rate by salary:')
print(analyzer.compute_turnover_rate_by_feature('salary'))

print('\nTurnover rate by department (sales):')
print(analyzer.compute_turnover_rate_by_feature('sales'))

---
## Step 3: Clustering of Employees Who Left

We apply **K-Means clustering (k=3)** exclusively to employees who left
(left==1), using two features: `satisfaction_level` and `last_evaluation`.

Expected cluster archetypes:
- **Burned-Out High Performers** — high evaluation, low satisfaction
- **Disengaged Low Performers** — low evaluation, low satisfaction  
- **Poached / Better Opportunity** — high evaluation, high satisfaction

### Cell 15 — K-Means Clustering (Step 3)

**Purpose:**
Groups the employees **who left** into 3 behavioural clusters using
K-Means. The goal is to understand *why* they left, not just that they did.
Each cluster will reveal a distinct departure archetype.

**Why only employees who left?**
Clustering the full workforce would mix stayers and leavers, diluting the
departure signal. We cluster only `left == 1` rows on two features:
`satisfaction_level` and `last_evaluation`.

**What this cell does — step by step:**
1. Defines `CLUSTER_OUTPUT` for saving the scatter plot.
2. Creates `EmployeeClusterer(df, n_clusters=3, output_dir=...)`.
3. Calls `run_clustering_pipeline()` which:
   a. Filters to `left == 1` (~3,571 employees).
   b. Fits `KMeans(n_clusters=3, random_state=42, n_init=10)`.
   c. Assigns a cluster label (0, 1, or 2) to each leaver.
   d. Computes per-cluster means and counts.
   e. Interprets each cluster using centroid heuristics.
   f. Saves a colour-coded scatter plot.
4. Prints the cluster summary table.

**Inputs:**

| Variable | Type | Description |
| ---------- | ---------- | ------------- |
| `df` | `pd.DataFrame` | Full HR DataFrame (from Cell 3) |
| `OUTPUT_DIR` | `Path` | Root output path |

**Outputs / Variables created:**

| Variable | Type | Description |
|----------|------|-------------|
| `CLUSTER_OUTPUT` | `Path` | `outputs/plots/clustering/` |
| `clusterer` | `EmployeeClusterer` | Fitted K-Means clusterer |
| `results` | `dict` | Keys: `summary` (DataFrame), `interpretation` (dict), `model` (KMeans) |

In [ ]:
# ─── K-Means Clustering ──────────────────────────────────────────────────────
CLUSTER_OUTPUT = OUTPUT_DIR / 'plots' / 'clustering'
clusterer = EmployeeClusterer(df, n_clusters=3, output_dir=CLUSTER_OUTPUT)

results = clusterer.run_clustering_pipeline()
print('Clustering complete.')
print(f'\nCluster Summary:')
print(results['summary'])

### Cell 16 — Cluster Interpretation & Visualisation

**Purpose:**
Translates the raw cluster numbers (0, 1, 2) into human-readable
**departure archetypes** and displays the scatter plot to visually confirm
the cluster separation.

**What this cell does — step by step:**
1. Iterates over `results['interpretation']` (computed in Cell 15).
2. For each cluster prints:
   - The **archetype label** (e.g., "Burned-Out High Performers")
   - Mean satisfaction and evaluation scores
   - Employee count in this cluster
   - A plain-English description of the group
3. Displays the K-Means scatter plot inline.

**Inputs:**

| Variable | Type | Description |
| ---------- | ---------- | ------------- |
| `results` | `dict` | Clustering results dict (from Cell 15) |
| `CLUSTER_OUTPUT` | `Path` | Path to saved cluster plot |

**Outputs:** Printed interpretation table + inline scatter plot.

**The three archetypes:**

| Archetype | Satisfaction | Evaluation | Retention action |
|-----------|-------------|------------|-----------------|
| Burned-Out High Performers | Low | High | Reduce workload, recognise effort |
| Disengaged Low Performers | Low | Low | PIP or managed exit |
| Poached / Better Opportunity | High | High | Competitive pay, growth path |

In [ ]:
# ─── Cluster Interpretation ───────────────────────────────────────────────────
print('Cluster Interpretation:')
print('=' * 60)
for cluster_id, info in results['interpretation'].items():
    print(f'\nCluster {cluster_id}: {info["label"]}')
    print(f'  Satisfaction mean : {info["satisfaction_mean"]}')
    print(f'  Evaluation mean   : {info["evaluation_mean"]}')
    print(f'  Count             : {info["count"]}')
    print(f'  Description: {info["description"][:100]}...')

display(Image(filename=str(CLUSTER_OUTPUT / 'kmeans_clusters.png')))

---
## Step 4: Handle Class Imbalance (SMOTE)

The dataset has a class imbalance — approximately 24% of employees left.
We address this with three preprocessing steps:
1. **One-hot encode** categorical columns (sales, salary) using pd.get_dummies
2. **Stratified 80:20 split** (random_state=123) preserving class proportions
3. **SMOTE** oversampling on the training set only

### Cell 18 — Class Distribution Before SMOTE

**Purpose:**
Quantifies the **class imbalance** in the raw dataset. About 24% of
employees left (`left = 1`) and 76% stayed (`left = 0`). If we train a
model on this imbalanced data without correction, it will be biased towards
predicting "stay" — achieving high accuracy by simply ignoring the minority
class.

**What this cell does — step by step:**
1. Creates a `DataPreprocessor` with `target_col='left'`,
   `test_size=0.2`, `random_state=123`.
2. Computes and prints raw counts and percentages for both classes.

**Inputs:**

| Variable | Type | Description |
| ---------- | ---------- | ------------- |
| `df` | `pd.DataFrame` | Full HR DataFrame (from Cell 3) |

**Outputs / Variables created:**

| Variable | Type | Description |
|----------|------|-------------|
| `preprocessor` | `DataPreprocessor` | Reused in Cell 19 to run the full pipeline |

**Expected output:**
- `stayed (0)`: ~11,428 employees (76.2%)
- `left   (1)`: ~3,571 employees (23.8%)

In [ ]:
# ─── Preprocessing & SMOTE ────────────────────────────────────────────────────
preprocessor = DataPreprocessor(df, target_col='left', test_size=0.2, random_state=123)

# Show class distribution before
print('Class distribution BEFORE SMOTE:')
print(f'  stayed (0): {(df["left"]==0).sum()} ({(df["left"]==0).mean()*100:.1f}%)')
print(f'  left   (1): {(df["left"]==1).sum()} ({(df["left"]==1).mean()*100:.1f}%)')

### Cell 19 — Full Preprocessing Pipeline: Encode → Split → SMOTE

**Purpose:**
Transforms the raw HR data into model-ready matrices by running three
sequential steps: one-hot encoding, stratified splitting, and SMOTE
oversampling.

**What this cell does — step by step:**

**Step 4.1 — One-Hot Encoding:**
Converts categorical text columns (`sales`, `salary`) into numeric
dummy variables using `pd.get_dummies`. For example, `salary = 'low'`
becomes three binary columns: `salary_low=1`, `salary_medium=0`,
`salary_high=0`.

**Step 4.2 — Stratified 80:20 Split:**
Splits data into 80% training (11,999 rows) and 20% test (3,000 rows).
"Stratified" means the 24%/76% class ratio is preserved in *both* partitions
— preventing the test set from accidentally having fewer leavers.

**Step 4.3 — SMOTE (Synthetic Minority Oversampling):**
Synthetically creates new `left=1` training examples by interpolating
between existing minority-class samples. Applied to the **training set
only** — the test set is never resampled so evaluation reflects real-world
proportions.

**Inputs:**

| Variable | Type | Description |
| ---------- | ---------- | ------------- |
| `preprocessor` | `DataPreprocessor` | `DataPreprocessor` instance (from Cell 18) |

**Outputs / Variables created:**

| Variable | Type | Description |
|----------|------|-------------|
| `X_train` | `pd.DataFrame` | SMOTE-balanced training features (~17,134 rows after SMOTE) |
| `X_test` | `pd.DataFrame` | Raw test features (3,000 rows, unmodified) |
| `y_train` | `pd.Series` | Balanced training labels (equal 0s and 1s after SMOTE) |
| `y_test` | `pd.Series` | Raw test labels (3,000 rows, original class ratio) |

In [ ]:
# ─── Run Full Preprocessing Pipeline ─────────────────────────────────────────
X_train, X_test, y_train, y_test = preprocessor.run_preprocessing_pipeline()

print('\nAfter preprocessing:')
print(f'  X_train shape  : {X_train.shape}')
print(f'  X_test shape   : {X_test.shape}')
print(f'  Feature columns: {len(X_train.columns)}')

print('\nClass distribution AFTER SMOTE (training set):')
print(f'  stayed (0): {(y_train==0).sum()}')
print(f'  left   (1): {(y_train==1).sum()}')

---
## Step 5: Model Training with 5-Fold Cross-Validation

We train three classifiers on the SMOTE-balanced training data,
each evaluated with **5-fold stratified cross-validation** using F1 as the
primary CV metric.

Models:
- **Logistic Regression** — interpretable linear baseline
- **Random Forest** — ensemble of decision trees, provides feature importance
- **Gradient Boosting** — sequential boosting, typically highest accuracy

### Cell 21 — Train All Models & Save to Disk (Step 5)

**Purpose:**
Trains three ML classifiers on the SMOTE-balanced training data, evaluates
each with 5-fold cross-validation, and persists every fitted model to disk
as a `.joblib` file for later use.

**Why three models?**
- **Logistic Regression** — interpretable linear baseline; establishes a
  minimum performance bar.
- **Random Forest** — ensemble of 100 decision trees; naturally handles
  non-linear interactions and provides feature importance.
- **Gradient Boosting** — builds trees sequentially, each correcting the
  errors of the last; typically highest accuracy on structured data.

**What this cell does — step by step:**
1. Defines `MODEL_OUTPUT` (plot directory) and `MODELS_DIR`
   (`outputs/models/`).
2. Creates `ModelTrainer(X_train, y_train, cv_folds=5)`.
3. Calls `trainer.train_all_models()` which for each classifier:
   a. Computes 5-fold stratified CV F1 scores (without fitting).
   b. Fits the model on the full training set.
4. Saves each fitted model as `<model_name>.joblib` in `outputs/models/`.

**Inputs:**

| Variable | Type | Description |
| ---------- | ---------- | ------------- |
| `X_train`, `y_train` | `—` | SMOTE-balanced training data (from Cell 19) |
| `X_test`, `y_test` | `—` | Test data for classification report plots |
| `OUTPUT_DIR` | `Path` | Root output path |

**Outputs / Variables created:**

| Variable / File | `—` | Description |
| ---------------- | ---------------- | ------------- |
| `trainer` | `ModelTrainer` | Fitted `ModelTrainer` instance |
| `models` | `dict` | Dict: `{'Logistic Regression': model, 'Random Forest': model, 'Gradient Boosting': model}` |
| `outputs/models/logistic_regression.joblib` | `—` | Serialised LR model |
| `outputs/models/random_forest.joblib` | `—` | Serialised RF model |
| `outputs/models/gradient_boosting.joblib` | Serialised GB model |

In [ ]:
# ─── Train All Models & Save to outputs/models/ ───────────────────────────────
import joblib

MODEL_OUTPUT = OUTPUT_DIR / 'plots' / 'modeling'
MODELS_DIR   = OUTPUT_DIR / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

trainer = ModelTrainer(X_train, y_train, cv_folds=5, output_dir=MODEL_OUTPUT)

print('Training models (this may take 1-2 minutes) ...')
models = trainer.train_all_models()
print(f'\nTrained models: {list(models.keys())}')

# Save every trained model to outputs/models/
print('\nSaving models to disk ...')
for name, model in models.items():
    safe_name = name.lower().replace(' ', '_')
    path = MODELS_DIR / f'{safe_name}.joblib'
    joblib.dump(model, path)
    print(f'  Saved: {path.name}')


### Cell 22 — 5-Fold Cross-Validation F1 Score Summary

**Purpose:**
Displays the **cross-validation performance** of all three models before
evaluating them on the held-out test set. CV scores are a more reliable
estimate of generalisation ability than a single train/test split because
they average performance across 5 different data splits.

**What this cell does — step by step:**
1. Iterates over each fitted model in `models`.
2. Calls `trainer.get_cv_scores(model)` which re-runs `cross_val_score`
   with `scoring='f1'` and `cv=5`.
3. Prints `mean ± std` for each model.

**Inputs:**

| Variable | Type | Description |
| ---------- | ---------- | ------------- |
| `models` | `dict` | Dict of fitted models (from Cell 21) |
| `trainer` | `ModelTrainer` | `ModelTrainer` instance (from Cell 21) |

**Outputs:** Printed table — no new variables created.

**How to read the output:**
- Higher mean = better average performance.
- Lower std = more consistent (less variance across folds).
- A large gap between CV score and test score later would indicate
  over- or under-fitting.

In [ ]:
# ─── CV Scores Summary ────────────────────────────────────────────────────────
print('5-Fold CV F1 Scores:')
print('-' * 50)
for name, model in models.items():
    scores = trainer.get_cv_scores(model)
    print(f'{name:25s}: {scores.mean():.4f} ± {scores.std():.4f}')

### Cell 23 — Classification Report Heatmaps

**Purpose:**
Generates a **heatmap visualisation** of Precision, Recall, and F1-score
for each model on the test set. These heatmaps make it easy to compare
per-class performance at a glance — especially the critical `Left (1)`
class performance.

**What this cell does — step by step:**
1. For each model, calls `trainer.plot_classification_report(model, X_test,
   y_test, name)` which:
   a. Generates predictions with `model.predict(X_test)`.
   b. Builds a `classification_report` dict (Stayed / Left / macro / weighted).
   c. Renders a `seaborn` heatmap with colour intensity representing metric
      values (0 = red, 1 = blue).
   d. Saves the PNG.
2. Displays the Gradient Boosting report inline as a representative example.

**Inputs:**

| Variable | Type | Description |
| ---------- | ---------- | ------------- |
| `models` | `dict` | Dict of fitted models |
| `trainer` | `ModelTrainer` | `ModelTrainer` instance |
| `X_test`, `y_test` | `—` | Held-out test data |

**Outputs:**

| File | Location |
|------|----------|
| `classification_report_logistic_regression.png` | `outputs/plots/modeling/` |
| `classification_report_random_forest.png` | `outputs/plots/modeling/` |
| `classification_report_gradient_boosting.png` | `outputs/plots/modeling/` |

In [ ]:
# ─── Classification Reports ──────────────────────────────────────────────────
for name, model in models.items():
    trainer.plot_classification_report(model, X_test, y_test, name)
    print(f'Classification report saved for: {name}')

# Display Gradient Boosting report as example
display(Image(filename=str(MODEL_OUTPUT / 'classification_report_gradient_boosting.png')))

---
## Step 6: Model Evaluation

We evaluate all three models on the held-out test set:
- **6.1** ROC/AUC curves (all models overlaid)
- **6.2** Confusion matrices per model
- **6.3** Select best model and justify Recall over Precision

### Cell 25 — Model Evaluation Setup (Step 6)

**Purpose:**
Initialises the `ModelEvaluator` object which will produce ROC curves,
confusion matrices, and the final model ranking table. Separating setup
from execution keeps subsequent cells clean and focused.

**What this cell does — step by step:**
1. Creates `ModelEvaluator(models, X_test, y_test, output_dir=MODEL_OUTPUT)`.
2. The evaluator stores references to all three fitted models and the
   test data — no computation happens yet.

**Inputs:**

| Variable | Type | Description |
| ---------- | ---------- | ------------- |
| `models` | `dict` | Dict of all three fitted models (from Cell 21) |
| `X_test`, `y_test` | `—` | Held-out test set (from Cell 19) |
| `MODEL_OUTPUT` | `Path` | Plot output directory (from Cell 21) |

**Outputs / Variables created:**

| Variable | Type | Description |
|----------|------|-------------|
| `evaluator` | `ModelEvaluator` | Reused across Cells 26–30 |

In [ ]:
# ─── Model Evaluation Setup ──────────────────────────────────────────────────
evaluator = ModelEvaluator(models, X_test, y_test, output_dir=MODEL_OUTPUT)

### Cell 26 — ROC / AUC Curves (Evaluation Step 6.1)

**Purpose:**
Plots overlaid **ROC (Receiver Operating Characteristic) curves** for all
three models. The ROC curve shows the trade-off between catching real
leavers (True Positive Rate / Recall) and incorrectly flagging stayers
(False Positive Rate) at every possible decision threshold.

**Understanding AUC:**
- **AUC = 1.0** → perfect model
- **AUC = 0.5** → random guessing (the diagonal dashed line)
- **AUC > 0.9** → excellent discrimination

**What this cell does — step by step:**
1. For each model, calls `model.predict_proba(X_test)[:, 1]` to get
   turnover probabilities.
2. Computes the ROC curve using `sklearn.metrics.roc_curve`.
3. Computes `roc_auc_score` and annotates it in the legend.
4. Overlays all three curves on one figure + the random baseline diagonal.
5. Saves the figure and displays it inline.

**Inputs:**

| Variable | Type | Description |
| ---------- | ---------- | ------------- |
| `evaluator` | `ModelEvaluator` | `ModelEvaluator` instance (from Cell 25) |

**Outputs:**

| File | Location |
|------|----------|
| `roc_curves.png` | `outputs/plots/modeling/` |
| Inline display | Notebook cell output |

In [ ]:
# ─── 6.1 ROC / AUC Curves ────────────────────────────────────────────────────
evaluator.plot_roc_curves()
print('ROC curves saved.')

display(Image(filename=str(MODEL_OUTPUT / 'roc_curves.png')))

### Cell 27 — Confusion Matrices (Evaluation Step 6.2)

**Purpose:**
Plots a **confusion matrix** for each model. The confusion matrix breaks
down all predictions into four buckets, making it crystal clear how many
employees were correctly identified as leavers vs. how many were missed.

**Reading a confusion matrix:**

|  | Predicted: Stay | Predicted: Leave |
|--|----------------|-----------------|
| **Actual: Stay** | True Negative (TN) ✓ | False Positive (FP) — unnecessary intervention |
| **Actual: Leave** | False Negative (FN) ✗ costly! | True Positive (TP) ✓ |

A **False Negative** (predicting "Stay" when the employee actually leaves)
is the most costly error — it means the HR team missed an at-risk employee.

**What this cell does — step by step:**
1. For each model: calls `model.predict(X_test)`, computes the 2×2 matrix
   with `sklearn.metrics.confusion_matrix`, renders a `seaborn` heatmap
   with raw counts annotated in each cell, and saves the PNG.
2. Displays all three confusion matrices inline.

**Inputs:**

| Variable | Type | Description |
| ---------- | ---------- | ------------- |
| `evaluator` | `ModelEvaluator` | `ModelEvaluator` instance |
| `models` | `dict` | Dict of fitted models (for iteration) |

**Outputs:**

| File | Location |
|------|----------|
| `confusion_matrix_logistic_regression.png` | `outputs/plots/modeling/` |
| `confusion_matrix_random_forest.png` | `outputs/plots/modeling/` |
| `confusion_matrix_gradient_boosting.png` | `outputs/plots/modeling/` |

In [ ]:
# ─── 6.2 Confusion Matrices ───────────────────────────────────────────────────
for name in models:
    evaluator.plot_confusion_matrix(name)
    print(f'Confusion matrix saved for: {name}')

# Display all confusion matrices
for name in models:
    safe_name = name.lower().replace(' ', '_')
    print(f'\n{name}:')
    display(Image(filename=str(MODEL_OUTPUT / f'confusion_matrix_{safe_name}.png')))

### Cell 28 — Model Performance Comparison Table

**Purpose:**
Generates a single consolidated table comparing all three models across
five metrics, sorted by AUC. This is the **primary decision-making table**
for choosing which model to deploy.

**What this cell does — step by step:**
1. Calls `evaluator.generate_evaluation_report()` which for each model:
   - Runs `predict` and `predict_proba` on `X_test`.
   - Computes AUC, Precision, Recall, F1, and Accuracy.
2. Returns a DataFrame sorted descending by AUC.
3. Prints the full table.

**Inputs:**

| Variable | Type | Description |
| ---------- | ---------- | ------------- |
| `evaluator` | `ModelEvaluator` | `ModelEvaluator` instance |

**Outputs / Variables created:**

| Variable | Type | Description |
|----------|------|-------------|
| `eval_report` | `pd.DataFrame` | Columns: Model, AUC, Precision, Recall, F1, Accuracy — sorted by AUC |

**Metric definitions:**

| Metric | Formula | What it measures |
|--------|---------|-----------------|
| AUC | Area under ROC curve | Overall discriminatory power across all thresholds |
| Recall | TP / (TP + FN) | Fraction of actual leavers correctly identified — **primary metric** |
| Precision | TP / (TP + FP) | Fraction of "leave" predictions that are correct |
| F1 | 2 × (P × R) / (P + R) | Harmonic mean of Precision and Recall |
| Accuracy | (TP + TN) / Total | Overall correctness (misleading with imbalanced classes) |

In [ ]:
# ─── Evaluation Report Table ─────────────────────────────────────────────────
eval_report = evaluator.generate_evaluation_report()
print('Model Performance Comparison (sorted by AUC):')
print(eval_report.to_string(index=False))

### Cell 29 — Best Model Selection & Persistence

**Purpose:**
Automatically identifies the **best performing model** (highest AUC) and
saves it separately as `best_model.joblib` so it can be loaded directly for
inference without re-training.

**What this cell does — step by step:**
1. Calls `evaluator.identify_best_model()` which:
   a. Computes AUC for each model.
   b. Returns the name and estimator of the highest-scoring model.
2. Serialises the best model to `outputs/models/best_model.joblib` using
   `joblib.dump()`.
3. Lists all `.joblib` files in `outputs/models/` to confirm all saves.

**Inputs:**

| Variable | Type | Description |
| ---------- | ---------- | ------------- |
| `evaluator` | `ModelEvaluator` | `ModelEvaluator` instance |
| `MODELS_DIR` | `Path` | `outputs/models/` directory path |

**Outputs / Variables created:**

| Variable / File | `—` | Description |
| ---------------- | ---------------- | ------------- |
| `best_name` | `str` | `str` — name of the winning model (typically "Random Forest") |
| `best_model` | `RandomForestClassifier` | Fitted sklearn estimator — reused in Step 7 |
| `outputs/models/best_model.joblib` | `—` | Serialised best model for production use |

**Expected winner:** Random Forest (AUC ≈ 0.9957 in the reference run).

In [ ]:
# ─── Best Model Selection & Save ─────────────────────────────────────────────
best_name, best_model = evaluator.identify_best_model()
print(f'Best model: {best_name}')

# Save best model separately for easy future loading
best_path = MODELS_DIR / 'best_model.joblib'
joblib.dump(best_model, best_path)
print(f'Best model saved: {best_path.name}')
print(f'\nAll models in {MODELS_DIR}:')
for f in sorted(MODELS_DIR.glob('*.joblib')):
    print(f'  {f.name}')


### Cell 30 — Why Recall is the Primary Metric (Step 6.3)

**Purpose:**
Provides the **business justification** for prioritising Recall over
Precision in the HR turnover context. This is not a technical default —
it is a deliberate decision driven by the asymmetric cost of errors.

**What this cell does:**
Calls `evaluator.justify_recall_over_precision()` which returns and prints
a pre-written explanation covering:

**The core argument:**

| Error type | What happens | Business cost |
|------------|-------------|---------------|
| False Negative (missed leaver) | Employee leaves undetected | 50–200% of annual salary in recruitment + onboarding |
| False Positive (flagged stayer) | Unnecessary retention conversation | Minor — a 30-minute check-in |

Since missing a real leaver costs orders of magnitude more than a false
alarm, we optimise for **Recall = TP / (TP + FN)** which directly minimises
missed leavers.

**Inputs:**

| Variable | Type | Description |
| ---------- | ---------- | ------------- |
| `evaluator` | `ModelEvaluator` | `ModelEvaluator` instance |

**Outputs:** Printed justification text only — no new variables.

In [ ]:
# ─── 6.3 Recall vs Precision Justification ───────────────────────────────────
print(evaluator.justify_recall_over_precision())

---
## Step 7: Retention Strategies

Using the best model, we:
1. Predict turnover probability for every employee in the test set
2. Assign each to one of four risk zones
3. Generate targeted HR retention strategies per zone

| Zone | Score | Priority |
|------|-------|----------|
| Safe (Green) | < 20% | Monitor |
| Low-Risk (Yellow) | 20–60% | Proactive |
| Medium-Risk (Orange) | 60–90% | Urgent |
| High-Risk (Red) | > 90% | Critical |

### Cell 32 — Retention Advisor Initialisation & Probability Prediction (Step 7)

**Purpose:**
Uses the best trained model to assign a **turnover probability score**
(0 to 100%) to every employee in the test set. These scores are the
foundation for assigning risk zones and tailoring retention interventions.

**What this cell does — step by step:**
1. Creates `RetentionAdvisor(best_model, X_test, y_test, output_dir=...)`.
2. Calls `advisor.predict_turnover_probabilities()` which runs
   `best_model.predict_proba(X_test)[:, 1]` — extracting the
   class-1 (leave) probability for each employee.
3. Prints min, mean, and max probability statistics.

**Inputs:**

| Variable | Type | Description |
| ---------- | ---------- | ------------- |
| `best_model` | `RandomForestClassifier` | Best fitted sklearn estimator (from Cell 29) |
| `X_test`, `y_test` | `—` | Held-out test data (from Cell 19) |
| `OUTPUT_DIR` | `Path` | Root output path |

**Outputs / Variables created:**

| Variable | Type | Description |
|----------|------|-------------|
| `RETENTION_OUTPUT` | `Path` | `outputs/plots/retention/` |
| `advisor` | `RetentionAdvisor` | Reused across Cells 33–36 |
| `probs` | `np.ndarray` | Per-employee turnover probabilities in [0, 1] |

In [ ]:
# ─── Retention Advisor ────────────────────────────────────────────────────────
RETENTION_OUTPUT = OUTPUT_DIR / 'plots' / 'retention'
advisor = RetentionAdvisor(best_model, X_test, y_test, output_dir=RETENTION_OUTPUT)

# Predict probabilities
probs = advisor.predict_turnover_probabilities()
print(f'Probability stats — min: {probs.min():.3f}, mean: {probs.mean():.3f}, max: {probs.max():.3f}')

### Cell 33 — Full Retention Report Generation

**Purpose:**
Combines turnover probabilities, risk zone assignments, and actual labels
into a single **per-employee retention report**. This is the operational
output that HR would use to prioritise interventions.

**Risk zone thresholds:**

| Zone | Probability | HR Priority |
|------|------------|-------------|
| Safe (Green) | < 20% | Monitor normally |
| Low-Risk (Yellow) | 20% – 60% | Proactive stay interview |
| Medium-Risk (Orange) | 60% – 90% | Immediate manager 1:1 |
| High-Risk (Red) | > 90% | Senior leadership intervention |

**What this cell does — step by step:**
1. Calls `advisor.generate_retention_report()` which:
   a. Calls `predict_turnover_probabilities()` (if not already done).
   b. Calls `categorize_risk_zones(probs)` using `pd.cut` on the four
      threshold boundaries.
   c. Maps each zone to a pre-written retention strategy string.
   d. Assembles a DataFrame with one row per employee.
2. Prints report shape and the zone distribution counts.
3. Displays the first 10 rows (without the verbose strategy text).

**Inputs:**

| Variable | Type | Description |
| ---------- | ---------- | ------------- |
| `advisor` | `RetentionAdvisor` | `RetentionAdvisor` instance (from Cell 32) |

**Outputs / Variables created:**

| Variable | Type | Description |
|----------|------|-------------|
| `retention_report` | `pd.DataFrame` | Columns: `employee_index`, `turnover_probability`, `risk_zone`, `actual_left`, `strategy` |

In [ ]:
# ─── Generate Full Retention Report ──────────────────────────────────────────
retention_report = advisor.generate_retention_report()
print(f'Retention report shape: {retention_report.shape}')
print('\nZone distribution:')
print(advisor.get_zone_counts())
print('\nFirst 10 rows (without strategy text):')
retention_report[['employee_index','turnover_probability','risk_zone','actual_left']].head(10)

### Cell 34 — Risk Zone Distribution Bar Chart

**Purpose:**
Visualises how the 3,000 test-set employees are distributed across the four
risk zones. This gives HR leadership an immediate picture of **how many
employees need attention** and at what urgency level.

**What this cell does — step by step:**
1. Calls `advisor.plot_zone_distribution()` which:
   a. Retrieves per-zone counts using `advisor.get_zone_counts()`.
   b. Draws a bar chart using zone-specific colours (green / yellow /
      orange / red).
   c. Annotates each bar with its exact count.
   d. Saves the figure.
2. Displays the saved PNG inline.

**Inputs:**

| Variable | Type | Description |
| ---------- | ---------- | ------------- |
| `advisor` | `RetentionAdvisor` | `RetentionAdvisor` with computed risk zones |
| `RETENTION_OUTPUT` | `Path` | Plot output directory |

**Outputs:**

| File | Location |
|------|----------|
| `risk_zone_distribution.png` | `outputs/plots/retention/` |
| Inline display | Notebook cell output |

**Expected distribution (reference run):**
Safe: 2,197 • Low-Risk: 96 • Medium-Risk: 60 • High-Risk: 647

In [ ]:
# ─── Zone Distribution Plot ───────────────────────────────────────────────────
advisor.plot_zone_distribution()
print('Zone distribution plot saved.')

display(Image(filename=str(RETENTION_OUTPUT / 'risk_zone_distribution.png')))

### Cell 35 — Retention Strategies per Risk Zone

**Purpose:**
Prints the **actionable HR intervention playbook** for each of the four
risk zones. These are concrete, prioritised recommendations — not generic
advice — tailored to the urgency level of each zone.

**What this cell does — step by step:**
1. Iterates over `ZONE_ORDER` = [Safe, Low-Risk, Medium-Risk, High-Risk].
2. For each zone: retrieves the employee count and calls
   `advisor.suggest_strategies(zone)` to return the pre-defined strategy
   text.
3. Prints a clearly labelled block per zone.

**Inputs:**

| Variable | Type | Description |
| ---------- | ---------- | ------------- |
| `advisor` | `RetentionAdvisor` | `RetentionAdvisor` with computed risk zones |
| `ZONE_ORDER` | `list[str]` | Ordered list of zone names (imported from `retention_advisor`) |

**Outputs:** Printed strategy text per zone — no new variables.

**Strategy summary:**

| Zone | Key HR Action |
|------|--------------|
| Safe | Annual reviews, recognition, optional stretch projects |
| Low-Risk | Quarterly stay interviews, career path clarification |
| Medium-Risk | Immediate 1:1, compensation review, bi-weekly check-ins |
| High-Risk | Senior leadership meeting, personalised retention package, weekly check-ins |

In [ ]:
# ─── Retention Strategies per Zone ───────────────────────────────────────────
for zone in ZONE_ORDER:
    count = advisor.get_zone_counts().get(zone, 0)
    print(f'\n{"="*60}')
    print(f'{zone.upper()} ZONE — {count} employees')
    print('=' * 60)
    print(advisor.suggest_strategies(zone))

### Cell 36 — Save Retention Report to CSV

**Purpose:**
Persists the per-employee retention report as a **CSV file** in the
`outputs/` directory. This is the deliverable for the HR team — they can
open it in Excel, import it into their HR system, or use it to filter and
sort employees by risk zone.

**What this cell does — step by step:**
1. Defines `report_path = outputs/retention_report.csv`.
2. Calls `retention_report.drop(columns=['strategy'])` to exclude the
   multi-line strategy text (which is not spreadsheet-friendly) before
   saving.
3. Writes the DataFrame to CSV without the row index.
4. Prints the confirmation path.

**Inputs:**

| Variable | Type | Description |
| ---------- | ---------- | ------------- |
| `retention_report` | `pd.DataFrame` | Full retention report DataFrame (from Cell 33) |
| `OUTPUT_DIR` | `Path` | Root output path |

**Outputs:**

| File | Columns |
|------|---------|
| `outputs/retention_report.csv` | `employee_index`, `turnover_probability`, `risk_zone`, `actual_left` |

> This file is committed to git (tracked in `outputs/`) and represents the
> final analytical output of the pipeline.

In [ ]:
# ─── Save Retention Report to CSV ────────────────────────────────────────────
report_path = OUTPUT_DIR / 'retention_report.csv'
retention_report.drop(columns=['strategy']).to_csv(report_path, index=False)
print(f'Retention report saved to: {report_path}')

---
## Summary & Conclusions

### Key Findings

1. **Data Quality**: The dataset is clean with no missing values. Duplicate rows may exist but do not affect model integrity significantly.

2. **EDA Insights**:
   - Employees with very few (2) or very many (6-7) projects have the highest turnover rates
   - Low satisfaction combined with high evaluation is a strong departure signal
   - Low salary employees show higher turnover rates

3. **Clustering**: Three distinct archetypes among departing employees:
   - **Burned-Out High Performers** — need workload relief and recognition
   - **Disengaged Low Performers** — need performance support or managed exits
   - **Poached / Better Opportunity** — need competitive compensation and growth paths

4. **Class Imbalance**: SMOTE successfully balanced the training set without
   distorting the test set evaluation.

5. **Model Performance**: Gradient Boosting typically achieves the highest AUC
   on structured HR data. All models outperform the random baseline significantly.

6. **Metric Choice**: **Recall** is the primary metric because missing an employee
   who will leave (false negative) costs 50-200% of their annual salary in
   replacement costs, vastly outweighing the cost of a false positive
   (unnecessary retention conversation).

7. **Retention Strategy**: Risk-zone segmentation enables HR to prioritise
   interventions — focusing immediate resources on Medium-Risk and High-Risk
   employees while maintaining engagement for Low-Risk and Safe employees.

### All Outputs Saved
- Plots: `outputs/plots/`
- Retention report: `outputs/retention_report.csv`
- Logs: `outputs/logs/`